# Kaggle Competition Template

Reusable starter notebook for a machine learning competition.

In [ ]:
# Core libraries
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

pd.set_option('display.max_columns', 200)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Data

Update the file paths and target column.

In [ ]:
# Competition inputs
DATA_DIR = Path('../input')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

display(train_df.head())
display(train_df.info())
display(train_df.describe(include='all').T.head(20))

## 2. Validation Strategy

Use a simple split for quick iteration, then replace with cross-validation if needed.

In [ ]:
target_column = 'target'  
id_column = 'id' 

X = train_df.drop(columns=[target_column])
y = train_df[target_column]

categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_columns = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_columns),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_columns),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
    sparse_threshold=0.3,
 )

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE,
    stratify=y if y.nunique() <= 20 else None,
 )

## 3. Train Baseline Model

Switch `task_type` between `classification` and `regression` as needed.

In [ ]:
task_type = 'classification'  

if task_type == 'classification':
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    scoring_name = 'ROC AUC' if y.nunique() == 2 else 'accuracy'
else:
    model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    scoring_name = 'MAE'

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model),
])

pipeline.fit(X_train, y_train)
valid_predictions = pipeline.predict(X_valid)

if task_type == 'classification':
    if y.nunique() == 2:
        valid_score = roc_auc_score(y_valid, pipeline.predict_proba(X_valid)[:, 1])
    else:
        valid_score = accuracy_score(y_valid, valid_predictions)
else:
    valid_score = mean_absolute_error(y_valid, valid_predictions)

print(f'Validation {scoring_name}: {valid_score:.5f}')

## 4. Predict and Submit

Write predictions to a Kaggle submission file.

In [ ]:
test_predictions = pipeline.predict(test_df)

submission = pd.DataFrame({
    id_column: test_df[id_column] if id_column in test_df.columns else np.arange(len(test_df)),
    target_column: test_predictions,
})

submission_path = Path('submission.csv')
submission.to_csv(submission_path, index=False)
submission.head()